In [ ]:
# =============================================================================
# CycleLayer MVP — Google Colab Training Script
# N-CMAPSS turbofan RUL prediction (physics-informed Brayton cycle layer)
#
# Compatible with: Colab (T4/A100/V100), Python 3.11+, PyTorch >= 2.2
# Usage: paste cells into a Colab notebook, or run as a script.
# =============================================================================

## Cell 0 — Quick-start checklist
1. Runtime > Change runtime type > **GPU** (T4 is fine)
2. Upload your HDF5 file to Google Drive, note its path
3. Edit the USER CONFIG block in Cell 2 (repo URL, HDF5 path, model type)
4. Run all cells in order

--- Cell: 1 — GPU / Environment Check ---

In [ ]:
import subprocess, sys, os, shutil, time, json, textwrap
from pathlib import Path
from datetime import datetime

def _sh(cmd, check=True, **kw):
    """Run shell command, stream output, raise on failure."""
    print(f"\n$ {cmd}")
    proc = subprocess.run(cmd, shell=True, check=check, **kw)
    return proc

# GPU check
try:
    _sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader", check=False)
except Exception:
    print("[WARN] nvidia-smi failed or no GPU found.")
import torch
print(f"\ntorch version : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM          : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[WARN] No GPU detected — training will be slow on CPU.")


# --- Cell: 2 — USER CONFIG (edit before running) ---

In [ ]:

# --------------------------------------------------------------------------
# Repo setup — choose ONE of the options below (A or B).
# Option A: Clone from GitHub (first run or if repo not yet mounted)
# Option B: Repo already available (mounted drive or previously cloned)
# --------------------------------------------------------------------------
REPO_URL   = "https://github.com/RobertKunte/cyclelayer-mvp.git"  # ← EDIT
REPO_CLONE = True          # Set False if repo already exists at REPO_ROOT

REPO_ROOT  = Path("/content/cyclelayer-mvp")   # where repo lives in Colab

# --------------------------------------------------------------------------
# Dataset path — HDF5 on Google Drive
# --------------------------------------------------------------------------
DRIVE_DATA_PATH = Path("/content/drive/MyDrive/cyclelayer-mvp/data/NCMAPSS/N-CMAPSS_DS01-005.h5")  # ← EDIT

# --------------------------------------------------------------------------
# Training settings
# --------------------------------------------------------------------------
MODEL_TYPE    = "cyclelayer_v1"   # "cyclelayer_v1" | "cnn" | "lstm" | "cnn_theta"
BASE_CONFIG   = "cyclelayer.yaml" # "cyclelayer.yaml" or "baseline.yaml"
STRIDE_TRAIN  = 5                 # 5 = dense (recommended); 10 = faster; 50 = quick test
STRIDE_EVAL   = 1                 # keep at 1 for correct PH computation
EPOCHS        = 50
PATIENCE      = 15
BATCH_SIZE    = 512               # reduce to 256 if OOM
NUM_WORKERS   = 4                 # Colab typically has 4 CPU cores
SEED          = 42

# --------------------------------------------------------------------------
# OOM guard — reduce batch size automatically on CUDA OOM
# --------------------------------------------------------------------------
OOM_FALLBACK_BATCH = 256          # used if BATCH_SIZE triggers OOM

# --------------------------------------------------------------------------
# CNN baseline comparison (Cell 8c)
# Trains a plain CNN on the same splits/sensor-scaler for fair comparison.
# Adds ~same wall-time as the main run (separate training loop).
# --------------------------------------------------------------------------
RUN_CNN_BASELINE = False          # set True to enable

# --------------------------------------------------------------------------
# LOUO cross-validation (Cell 9)
# Runs 6 leave-one-unit-out folds for the main model — the proper
# generalisation estimate.  Takes ~6× the single-run wall-time.
# --------------------------------------------------------------------------
RUN_LOUO          = False         # set True to enable LOUO for main model
RUN_LOUO_BASELINE = False         # set True to also run LOUO for CNN baseline
                                  # (requires RUN_CNN_BASELINE = True)

# --------------------------------------------------------------------------
# Artifact destinations
# --------------------------------------------------------------------------
RUNS_ROOT        = Path("/content/runs")
DRIVE_RUNS_ROOT  = Path("/content/drive/MyDrive/cyclelayer_runs")  # ← EDIT (or None)

print("Config:")
for k, v in dict(
    MODEL_TYPE=MODEL_TYPE, BASE_CONFIG=BASE_CONFIG,
    STRIDE_TRAIN=STRIDE_TRAIN, EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE, SEED=SEED,
    RUN_CNN_BASELINE=RUN_CNN_BASELINE,
    RUN_LOUO=RUN_LOUO, RUN_LOUO_BASELINE=RUN_LOUO_BASELINE,
).items():
    print(f"  {k:22s} = {v}")


In [ ]:
_sh("pip install -q -U pip")
_sh(
    "pip install -q "
    "torch>=2.2 "
    "numpy>=1.26 "
    "scipy>=1.12 "
    "h5py>=3.10 "
    "matplotlib>=3.8 "
    "pandas>=2.2 "
    "tensorboard>=2.16 "
    "pyyaml>=6.0 "
    "tqdm>=4.66"
)
print("Dependencies installed.")


# --- Cell: 4 — Clone / Load Repo ---

In [ ]:
import shutil
if REPO_CLONE:
    if REPO_ROOT.exists():
        print(f"Repo already exists at {REPO_ROOT}. Pulling latest...")
        try:
            _sh(f"git -C {REPO_ROOT} pull --ff-only")
        except Exception:
            print("[WARN] Pull failed. Force re-cloning...")
            shutil.rmtree(REPO_ROOT, ignore_errors=True)
            _sh(f"git clone {REPO_URL} {REPO_ROOT}")
    else:
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        _sh(f"git clone {REPO_URL} {REPO_ROOT}")
else:
    assert REPO_ROOT.exists(), f"REPO_ROOT not found: {REPO_ROOT}"
    print(f"Using existing repo at {REPO_ROOT}")

# Make sure repo is importable
for p in [str(REPO_ROOT / "src"), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Verify key entry points exist
for script in ["scripts/train.py", "scripts/evaluate.py", "scripts/run_louo.py"]:
    assert (REPO_ROOT / script).exists(), f"Missing: {script}"

print(f"\nGit commit: ", end="")
_sh(f"git -C {REPO_ROOT} rev-parse --short HEAD")

# ---------------------------------------------------------------------------
# Recover src/cyclelayer/data/ if it was excluded by .gitignore
#
# Background: .gitignore contains the pattern "data/" which accidentally
# matches src/cyclelayer/data/ (the Python package) in addition to the
# intended ./data/ directory (large HDF5 files).
#
# Recovery strategy (in order of preference):
#   1. Files are already present (git was fixed or repo was not affected)
#   2. Copy from Google Drive  (upload src/cyclelayer/data/ there once)
#   3. Raise a clear error with instructions
# ---------------------------------------------------------------------------
_data_pkg = REPO_ROOT / "src" / "cyclelayer" / "data"
_data_pkg_ok = (
    _data_pkg.exists()
    and (_data_pkg / "__init__.py").exists()
    and (_data_pkg / "ncmapss.py").exists()
    and (_data_pkg / "splits.py").exists()
    and (_data_pkg / "preprocessing.py").exists()
)

if not _data_pkg_ok:
    print("\n[WARN] src/cyclelayer/data/ is incomplete in the cloned repo.")
    print("       This is caused by .gitignore 'data/' matching the Python package.")

    # Try to restore from Google Drive
    _drive_src = Path("/content/drive/MyDrive/cyclelayer-mvp/src/cyclelayer/data")
    if _drive_src.exists() and (_drive_src / "ncmapss.py").exists():
        print(f"       Restoring from Drive: {_drive_src} ...")
        _data_pkg.mkdir(parents=True, exist_ok=True)
        for _f in _drive_src.glob("*.py"):
            shutil.copy2(_f, _data_pkg / _f.name)
        print(f"       Restored: {[f.name for f in _data_pkg.glob('*.py')]}")
    else:
        raise RuntimeError(
            "\n"
            "src/cyclelayer/data/ is missing from the cloned repo and from Drive.\n"
            "\n"
            "PERMANENT FIX (recommended):\n"
            "  In .gitignore, change the line:\n"
            "      data/\n"
            "  to:\n"
            "      /data/\n"
            "  This anchors the rule to the repo root, so the HDF5 files in ./data/\n"
            "  remain excluded, but src/cyclelayer/data/*.py (Python source, ~20 KB)\n"
            "  can be committed. Then: git add src/cyclelayer/data/ && git push\n"
            "\n"
            "QUICK WORKAROUND (no git changes required):\n"
            f"  Upload the folder src/cyclelayer/data/ from your local machine to:\n"
            f"  Google Drive -> cyclelayer-mvp/src/cyclelayer/data/\n"
            f"  (copy the 4 .py files there, then re-run this cell)\n"
        )
else:
    print("src/cyclelayer/data/ OK:", [f.name for f in _data_pkg.glob("*.py")])

# Install package so subprocess Python can also import it
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "."],
    cwd=str(REPO_ROOT), check=True,
)
print("Package installed (editable).")


In [ ]:
from google.colab import drive  # noqa: E402  (Colab-specific)
drive.mount("/content/drive", force_remount=False)

assert DRIVE_DATA_PATH.exists(), (
    f"\nHDF5 file not found: {DRIVE_DATA_PATH}\n"
    "Ensure the file is uploaded to Google Drive at the path set in USER CONFIG."
)

file_size_gb = DRIVE_DATA_PATH.stat().st_size / 1e9
print(f"HDF5 found: {DRIVE_DATA_PATH}  ({file_size_gb:.2f} GB)")

# Quick sanity check — list top-level keys
import h5py
with h5py.File(DRIVE_DATA_PATH, "r") as f:
    keys = list(f.keys())
    print(f"HDF5 keys : {keys}")
    assert any("dev" in k for k in keys), (
        f"Expected keys ending in '_dev'. Found: {keys}"
    )
print("HDF5 structure OK.")


# --- Cell: 6 — Build Patched Config ---

In [ ]:

# --- Cell: 5b — HDF5 Inspector (optional but recommended) ---
# Prints dataset shapes, dtypes, unit IDs, and the key naming convention.
# Runs fast — no full array loads.

_sh(
    f"{sys.executable} scripts/inspect_hdf5.py "
    f"{DRIVE_DATA_PATH} --units",
    cwd=REPO_ROOT,
)


In [ ]:
import yaml

RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_{MODEL_TYPE}"
RUN_DIR = RUNS_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=True)

RUNTIME_CFG_DIR = Path("/content/configs_runtime")
RUNTIME_CFG_DIR.mkdir(parents=True, exist_ok=True)

# Load base config
base_cfg_path = REPO_ROOT / "configs" / BASE_CONFIG
with open(base_cfg_path) as f:
    cfg = yaml.safe_load(f)

# --- Override data section ---
cfg.setdefault("data", {})
cfg["data"]["hdf5_path"]      = str(DRIVE_DATA_PATH)
cfg["data"]["stride_train"]   = STRIDE_TRAIN
cfg["data"]["stride_eval"]    = STRIDE_EVAL
cfg["data"]["batch_size"]     = BATCH_SIZE
cfg["data"]["num_workers"]    = NUM_WORKERS

# --- Override model section ---
cfg.setdefault("model", {})
cfg["model"]["type"] = MODEL_TYPE

# --- Override training section ---
cfg.setdefault("training", {})
cfg["training"]["epochs"]                  = EPOCHS
cfg["training"]["early_stopping_patience"] = PATIENCE
cfg["training"]["output_dir"]              = str(RUN_DIR)
cfg["training"]["use_amp"]                 = torch.cuda.is_available()

# --- Override evaluation section ---
cfg.setdefault("evaluation", {})
cfg["evaluation"]["checkpoint"] = str(RUN_DIR / "best.pt")

# --- Seed ---
cfg["seed"] = SEED

# Save patched config
patched_cfg_name = f"cyclelayer_{MODEL_TYPE}_stride{STRIDE_TRAIN}.yaml"
PATCHED_CFG = RUNTIME_CFG_DIR / patched_cfg_name
with open(PATCHED_CFG, "w") as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print(f"Patched config: {PATCHED_CFG}")
print(f"Run dir       : {RUN_DIR}")
print("\nKey overrides:")
print(f"  hdf5_path    : {cfg['data']['hdf5_path']}")
print(f"  stride_train : {cfg['data']['stride_train']}")
print(f"  stride_eval  : {cfg['data']['stride_eval']}")
print(f"  batch_size   : {cfg['data']['batch_size']}")
print(f"  model.type   : {cfg['model']['type']}")
print(f"  epochs       : {cfg['training']['epochs']}")
print(f"  use_amp      : {cfg['training']['use_amp']}")
print(f"  output_dir   : {cfg['training']['output_dir']}")


# --- Cell: 7 — Training ---

In [ ]:

def _stream_subprocess(cmd: list[str], logfile: Path, cwd: Path) -> None:
    """Run subprocess, print output live, and tee to logfile."""
    logfile.parent.mkdir(parents=True, exist_ok=True)
    print(f"Running: {' '.join(cmd)}\nLog: {logfile}\n{'='*70}")
    t0 = time.time()

    with open(logfile, "w", encoding="utf-8") as log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="replace",
            cwd=str(cwd),
            env=dict(os.environ, PYTHONPATH=f"{cwd}/src:{os.environ.get('PYTHONPATH', '')}"),
        )
        for line in proc.stdout:
            print(line, end="", flush=True)
            log.write(line)
        proc.wait()

    elapsed = time.time() - t0
    print(f"\n{'='*70}")
    print(f"Elapsed: {elapsed/60:.1f} min  |  Return code: {proc.returncode}")

    if proc.returncode != 0:
        # OOM fallback: retry with smaller batch
        if "CUDA out of memory" in open(logfile, encoding="utf-8").read():
            print(f"\n[OOM] Retrying with batch_size={OOM_FALLBACK_BATCH} ...")
            cfg["data"]["batch_size"] = OOM_FALLBACK_BATCH
            with open(PATCHED_CFG, "w") as f:
                yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
            _stream_subprocess(cmd, logfile.with_suffix(".retry.log"), cwd)
        else:
            raise RuntimeError(f"Command failed (code {proc.returncode}). See {logfile}")


LOG_DIR = RUN_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

train_cmd = [
    sys.executable, "scripts/train.py",
    "--config", str(PATCHED_CFG),
    "--model", MODEL_TYPE,
]

_stream_subprocess(train_cmd, LOG_DIR / "train.log", cwd=REPO_ROOT)

# Verify checkpoint produced
assert (RUN_DIR / "best.pt").exists(), (
    f"Training did not produce best.pt in {RUN_DIR}. Check {LOG_DIR}/train.log"
)
print(f"\nCheckpoint: {RUN_DIR / 'best.pt'} ({(RUN_DIR / 'best.pt').stat().st_size / 1e6:.1f} MB)")


# --- Cell: 8 — Evaluation (test split + dev split) ---

In [ ]:

def _evaluate(split: str, logfile: Path) -> dict:
    """Run evaluate.py for one split, return loaded metrics dict."""
    metrics_path = RUN_DIR / f"metrics_{split}.json"
    eval_cmd = [
        sys.executable, "scripts/evaluate.py",
        "--config",     str(PATCHED_CFG),
        "--checkpoint", str(RUN_DIR / "best.pt"),
        "--split",      split,
        "--output",     str(metrics_path),
    ]
    _stream_subprocess(eval_cmd, logfile, cwd=REPO_ROOT)

    # predictions.csv is written next to metrics.json by evaluate.py
    # rename it to avoid collision between splits
    default_pred = RUN_DIR / "predictions.csv"
    split_pred   = RUN_DIR / f"predictions_{split}.csv"
    if default_pred.exists() and not split_pred.exists():
        default_pred.rename(split_pred)

    if metrics_path.exists():
        with open(metrics_path) as f:
            return json.load(f)
    return {}


# Evaluate on held-out dev (val + test units)
metrics_dev = _evaluate("dev", LOG_DIR / "eval_dev.log")

# Evaluate on the HDF5 test split (fully independent rows)
metrics_test = _evaluate("test", LOG_DIR / "eval_test.log")

print("\n" + "="*60)
print("EVALUATION SUMMARY")
print("="*60)
for split_name, m in [("dev", metrics_dev), ("test", metrics_test)]:
    if m:
        print(f"\n  Split: {split_name}")
        print(f"    RMSE       : {m.get('rmse', 'n/a'):.4f}")
        print(f"    S-score    : {m.get('s_score', 'n/a'):.2f}")
        print(f"    PH median  : {m.get('ph_median', 'n/a')}")
        print(f"    PH p10/p90 : {m.get('ph_p10', 'n/a')} / {m.get('ph_p90', 'n/a')}")
        print(f"    PH none    : {m.get('ph_none_count', 'n/a')}")


# --- Cell: 9 — (Optional) LOUO Cross-Validation ---

In [ ]:

# --- Cell: 9 — LOUO Cross-Validation (optional) ---
# Leave-One-Unit-Out: trains 6 folds (each holds out one engine unit) to get
# the proper generalisation estimate.  Controls: RUN_LOUO / RUN_LOUO_BASELINE in Cell 2.
# Wall-time: ~6× a single run per model.

louo_results      = None
louo_results_cnn  = None


def _run_louo(model_type: str, cfg_source: Path, parent_run_dir: Path) -> dict | None:
    """Run LOUO for one model type; return aggregated results dict or None."""
    louo_dir = parent_run_dir / f"louo_{model_type}"

    cfg_louo = yaml.safe_load(open(cfg_source))
    # Anchor trick: run_louo.py creates {parent(output_dir)}/louo_{model}/
    cfg_louo["training"]["output_dir"] = str(parent_run_dir / "_louo_anchor")
    louo_cfg_path = RUNTIME_CFG_DIR / f"louo_{model_type}_stride{STRIDE_TRAIN}.yaml"
    with open(louo_cfg_path, "w") as f:
        yaml.dump(cfg_louo, f, default_flow_style=False, sort_keys=False)

    _stream_subprocess(
        [sys.executable, "scripts/run_louo.py",
         "--config", str(louo_cfg_path),
         "--model",  model_type],
        parent_run_dir / "logs" / f"louo_{model_type}.log",
        cwd=REPO_ROOT,
    )

    results_json = louo_dir / "results.json"
    if results_json.exists():
        return json.load(open(results_json))
    print(f"[WARN] {results_json} not found — LOUO may have failed.")
    return None


def _print_louo(label: str, r: dict) -> None:
    print(f"\n  {label}")
    print(f"    RMSE    : {r.get('rmse_mean',    float('nan')):.4f} ± {r.get('rmse_std',    float('nan')):.4f}")
    print(f"    S-score : {r.get('s_score_mean', float('nan')):.2f}   ± {r.get('s_score_std', float('nan')):.2f}")
    print(f"    PH med  : {r.get('ph_median',    'NaN')}")
    print(f"    PH none : {r.get('ph_none_count','?')}/{r.get('n_folds','?')} folds")


# ── Main model LOUO ──────────────────────────────────────────────────────────
if RUN_LOUO:
    print(f"Running LOUO for {MODEL_TYPE} ...")
    louo_results = _run_louo(MODEL_TYPE, PATCHED_CFG, RUN_DIR)
else:
    print(f"LOUO for {MODEL_TYPE} skipped (RUN_LOUO=False in Cell 2).")

# ── CNN baseline LOUO ────────────────────────────────────────────────────────
if RUN_LOUO_BASELINE:
    if not RUN_CNN_BASELINE or CNN_CFG is None or RUN_DIR_CNN is None:
        print("\n[WARN] RUN_LOUO_BASELINE=True but CNN baseline was not run "
              "(RUN_CNN_BASELINE=False). Skipping.")
    else:
        print(f"\nRunning LOUO for CNN baseline ...")
        louo_results_cnn = _run_louo("cnn", CNN_CFG, RUN_DIR_CNN)
else:
    if RUN_CNN_BASELINE:
        print("LOUO for CNN baseline skipped (RUN_LOUO_BASELINE=False in Cell 2).")

# ── Summary ───────────────────────────────────────────────────────────────────
if louo_results or louo_results_cnn:
    print("\n" + "=" * 72)
    print("LOUO RESULTS")
    print("=" * 72)
    if louo_results:
        _print_louo(f"{MODEL_TYPE} (main)", louo_results)
    if louo_results_cnn:
        _print_louo("CNN baseline", louo_results_cnn)

    # Side-by-side comparison if both ran
    if louo_results and louo_results_cnn:
        print(f"\n  {'Metric':<15}  {'main':>10}  {'cnn':>10}  {'Δ':>10}")
        print("  " + "-" * 50)
        for key, label in [("rmse_mean", "RMSE"), ("s_score_mean", "S-score"), ("ph_none_count", "PH none")]:
            v_m = louo_results.get(key, float("nan"))
            v_c = louo_results_cnn.get(key, float("nan"))
            try:
                delta = float(v_m) - float(v_c)
                print(f"  {label:<15}  {v_m:>10.3f}  {v_c:>10.3f}  {delta:>+10.3f}")
            except Exception:
                print(f"  {label:<15}  {str(v_m):>10}  {str(v_c):>10}  {'—':>10}")
        print("  Δ = main − cnn  ·  negative = main model better (for RMSE/S-score/PH-none)")
    print("=" * 72)


In [ ]:

# --- Cell: 8b — Diagnostic Plots (analyze_predictions.py) ---
# Generates trajectory plots, error histograms, scatter, spike locator CSV,
# and a self-contained report.md explaining all metrics.
# Uses matplotlib Agg — no display needed (Colab compatible).

ANALYSIS_DIR = RUN_DIR / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)

# Analyse both splits if CSVs were produced
for split_name in ["dev", "test"]:
    csv_path = RUN_DIR / f"predictions_{split_name}.csv"
    if not csv_path.exists():
        # Fallback: predictions.csv without suffix (single-split runs)
        csv_path = RUN_DIR / "predictions.csv"
    if not csv_path.exists():
        print(f"[SKIP] No predictions CSV found for split={split_name}")
        continue

    split_out = ANALYSIS_DIR / split_name
    split_out.mkdir(exist_ok=True)

    _stream_subprocess(
        [
            sys.executable, "scripts/analyze_predictions.py",
            "--pred_csv",  str(csv_path),
            "--out_dir",   str(split_out),
            "--n_spikes",  "20",
        ],
        logfile=LOG_DIR / f"analyze_{split_name}.log",
        cwd=REPO_ROOT,
    )
    print(f"\nDiagnostic outputs for split={split_name}:")
    for f in sorted(split_out.iterdir()):
        print(f"  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)")

# Quick inline display of the dev trajectory for the first unit (if Colab)
try:
    from IPython.display import Image, display
    first_png = next((ANALYSIS_DIR / "dev").glob("unit_*_trajectory.png"), None)
    if first_png:
        print(f"\nShowing: {first_png.name}")
        display(Image(str(first_png)))
except Exception:
    pass  # not in Colab or IPython — plots are saved to disk

print(f"\nAll diagnostic files in: {ANALYSIS_DIR}")


In [ ]:

# --- Cell: 8c — CNN Baseline Comparison (optional) ---
# Trains a plain CNN on the SAME unit splits and sensor scaler as the main run,
# then prints a side-by-side metric table.  Controls: RUN_CNN_BASELINE in Cell 2.

metrics_cnn_dev  = {}
metrics_cnn_test = {}
RUN_DIR_CNN      = None
CNN_CFG          = None

if RUN_CNN_BASELINE:
    # ── Build baseline config ────────────────────────────────────────────────
    baseline_cfg_path = REPO_ROOT / "configs" / "baseline.yaml"
    with open(baseline_cfg_path) as f:
        cfg_cnn = yaml.safe_load(f)

    RUN_DIR_CNN = RUNS_ROOT / (RUN_ID + "_cnn")
    RUN_DIR_CNN.mkdir(parents=True, exist_ok=True)

    cfg_cnn["data"]["hdf5_path"]    = str(DRIVE_DATA_PATH)
    cfg_cnn["data"]["stride_train"] = STRIDE_TRAIN
    cfg_cnn["data"]["stride_eval"]  = STRIDE_EVAL
    cfg_cnn["data"]["batch_size"]   = BATCH_SIZE
    cfg_cnn["data"]["num_workers"]  = NUM_WORKERS
    # Reuse the same unit splits produced by the main run (seed=42, deterministic)
    cfg_cnn["data"]["split_dir"] = str(
        REPO_ROOT / "splits" / DRIVE_DATA_PATH.stem
    )
    cfg_cnn["model"]["type"]                       = "cnn"
    cfg_cnn["training"]["epochs"]                  = EPOCHS
    cfg_cnn["training"]["early_stopping_patience"] = PATIENCE
    cfg_cnn["training"]["output_dir"]              = str(RUN_DIR_CNN)
    cfg_cnn["training"]["use_amp"]                 = torch.cuda.is_available()
    cfg_cnn["evaluation"]["checkpoint"]            = str(RUN_DIR_CNN / "best.pt")
    cfg_cnn["seed"]                                = SEED

    CNN_CFG = RUNTIME_CFG_DIR / f"baseline_cnn_stride{STRIDE_TRAIN}.yaml"
    with open(CNN_CFG, "w") as f:
        yaml.dump(cfg_cnn, f, default_flow_style=False, sort_keys=False)

    LOG_DIR_CNN = RUN_DIR_CNN / "logs"
    LOG_DIR_CNN.mkdir(exist_ok=True)
    print(f"CNN baseline config : {CNN_CFG}")
    print(f"CNN baseline run dir: {RUN_DIR_CNN}\n")

    # ── Train ────────────────────────────────────────────────────────────────
    _stream_subprocess(
        [sys.executable, "scripts/train.py",
         "--config", str(CNN_CFG), "--model", "cnn"],
        LOG_DIR_CNN / "train.log",
        cwd=REPO_ROOT,
    )
    assert (RUN_DIR_CNN / "best.pt").exists(), (
        f"CNN baseline training did not produce best.pt. "
        f"See {LOG_DIR_CNN}/train.log"
    )

    # ── Evaluate ─────────────────────────────────────────────────────────────
    def _evaluate_cnn(split: str) -> dict:
        out_path = RUN_DIR_CNN / f"metrics_{split}.json"
        _stream_subprocess(
            [sys.executable, "scripts/evaluate.py",
             "--config",     str(CNN_CFG),
             "--checkpoint", str(RUN_DIR_CNN / "best.pt"),
             "--split",      split,
             "--output",     str(out_path)],
            LOG_DIR_CNN / f"eval_{split}.log",
            cwd=REPO_ROOT,
        )
        pred_csv = RUN_DIR_CNN / "predictions.csv"
        if pred_csv.exists() and not (RUN_DIR_CNN / f"predictions_{split}.csv").exists():
            pred_csv.rename(RUN_DIR_CNN / f"predictions_{split}.csv")
        return json.load(open(out_path)) if out_path.exists() else {}

    metrics_cnn_dev  = _evaluate_cnn("dev")
    metrics_cnn_test = _evaluate_cnn("test")

    # ── Diagnostic plots ─────────────────────────────────────────────────────
    ANALYSIS_DIR_CNN = RUN_DIR_CNN / "analysis"
    ANALYSIS_DIR_CNN.mkdir(exist_ok=True)
    for split_name in ["dev", "test"]:
        csv_path = RUN_DIR_CNN / f"predictions_{split_name}.csv"
        if not csv_path.exists():
            continue
        split_out = ANALYSIS_DIR_CNN / split_name
        split_out.mkdir(exist_ok=True)
        _stream_subprocess(
            [sys.executable, "scripts/analyze_predictions.py",
             "--pred_csv", str(csv_path),
             "--out_dir",  str(split_out),
             "--n_spikes", "20"],
            LOG_DIR_CNN / f"analyze_{split_name}.log",
            cwd=REPO_ROOT,
        )

    # ── Side-by-side comparison ───────────────────────────────────────────────
    print("\n" + "=" * 72)
    print(f"COMPARISON  ·  {MODEL_TYPE}  vs  CNN baseline  ·  stride={STRIDE_TRAIN}")
    print("=" * 72)
    METRICS_TO_COMPARE = [
        ("rmse",            "RMSE",          True),   # lower is better
        ("s_score_mean",    "S-score/win",   True),
        ("max_abs_error_unit_median", "MaxAbsErr",  True),
        ("ph_median",       "PH median",     False),  # higher is better / NaN ok
        ("ph_none_count",   "PH none",       True),
    ]
    hdr = f"{'Split·Metric':<22}  {'main':>10}  {'cnn':>10}  {'Δ':>10}  {'Δ%':>7}"
    print(hdr)
    print("-" * 72)
    for split_name, m_main, m_cnn in [
        ("dev",  metrics_dev,  metrics_cnn_dev),
        ("test", metrics_test, metrics_cnn_test),
    ]:
        for key, label, lower_better in METRICS_TO_COMPARE:
            v_m = m_main.get(key, float("nan"))
            v_c = m_cnn.get(key,  float("nan"))
            try:
                delta     = float(v_m) - float(v_c)
                delta_pct = 100.0 * delta / abs(float(v_c)) if float(v_c) != 0 else float("nan")
                # positive delta = main worse (for lower-is-better metrics)
                flag = "✓" if (lower_better and delta < 0) or (not lower_better and delta > 0) else "✗"
                print(f"{split_name+'/'+label:<22}  {v_m:>10.3f}  {v_c:>10.3f}  "
                      f"{delta:>+10.3f}  {delta_pct:>+6.1f}%  {flag}")
            except Exception:
                print(f"{split_name+'/'+label:<22}  {str(v_m):>10}  {str(v_c):>10}  {'—':>10}  {'—':>7}")
    print("=" * 72)
    print("✓ = main model better  ·  Δ = main − cnn  ·  lower is better for RMSE/S-score/Err/PH-none")
    print(f"\nCNN run dir: {RUN_DIR_CNN}")
    try:
        from IPython.display import Image, display
        for split_name in ["test", "dev"]:
            scatter = (ANALYSIS_DIR_CNN / split_name / "scatter_pred_vs_true.png")
            if scatter.exists():
                print(f"\nCNN baseline scatter ({split_name}):")
                display(Image(str(scatter)))
    except Exception:
        pass
else:
    print("CNN baseline skipped (RUN_CNN_BASELINE=False in Cell 2).")


In [ ]:

def _fmt_size(path: Path) -> str:
    if path.exists():
        return f"{path.stat().st_size / 1e6:.2f} MB"
    return "(missing)"

print(f"\nArtifacts in: {RUN_DIR}")
print(f"{'File':<40} {'Size':>10}")
print("-" * 52)
for fname in [
    "best.pt", "last.pt",
    "theta_scaler.npz",
    "metrics_dev.json", "metrics_test.json",
    "predictions_dev.csv", "predictions_test.csv",
    "logs/train.log", "logs/eval_dev.log", "logs/eval_test.log",
]:
    p = RUN_DIR / fname
    status = _fmt_size(p) if p.exists() else "(absent)"
    print(f"  {fname:<38} {status:>10}")

# List TensorBoard event files
tb_dir = RUN_DIR / "tb"
if tb_dir.exists():
    tb_files = list(tb_dir.glob("events.out.*"))
    print(f"  tb/events.out.*  ({len(tb_files)} file(s))")

# Config and meta
print(f"\n  Config used  : {PATCHED_CFG}")


# --- Cell: 11 — Package & Copy to Google Drive ---

In [ ]:
if DRIVE_RUNS_ROOT is not None:
    DRIVE_RUNS_ROOT.mkdir(parents=True, exist_ok=True)

    # Create zip archive of the run directory
    zip_base = str(DRIVE_RUNS_ROOT / RUN_ID)
    print(f"Zipping {RUN_DIR} -> {zip_base}.zip ...")
    zip_path = shutil.make_archive(
        zip_base,
        format="zip",
        root_dir=str(RUN_DIR.parent),
        base_dir=str(RUN_DIR.name),
    )
    zip_size_mb = Path(zip_path).stat().st_size / 1e6
    print(f"Created: {zip_path}  ({zip_size_mb:.1f} MB)")

    # Also copy patched config alongside the zip for reference
    shutil.copy2(PATCHED_CFG, DRIVE_RUNS_ROOT / PATCHED_CFG.name)
    print(f"Config copy: {DRIVE_RUNS_ROOT / PATCHED_CFG.name}")
else:
    print("DRIVE_RUNS_ROOT is None — skipping Drive upload.")
    print("To save artifacts manually:")
    print(f"  from google.colab import files")
    print(f"  files.download('{RUN_DIR}/best.pt')")


# --- Cell: 12 — Optional: View TensorBoard ---

In [ ]:
# Uncomment and run to launch TensorBoard in Colab:

%load_ext tensorboard
%tensorboard --logdir /content/runs

# Or equivalently:
# _sh(f"tensorboard --logdir {RUNS_ROOT} --port 6006 &")
# from google.colab.output import eval_js
# print(eval_js("google.colab.kernel.proxyPort(6006)"))


# --- Cell: 13 — Troubleshooting Notes ---

In [ ]:

NOTES = textwrap.dedent("""
    TROUBLESHOOTING
    ===============

    1. OOM (CUDA out of memory)
       - The script automatically retries with batch_size={fallback} on OOM.
       - Manually: set BATCH_SIZE = 256 or 128 in Cell 2.
       - Last resort: add  cfg["data"]["window_size"] = 20  in Cell 6.

    2. HDF5 file not found
       - Verify DRIVE_DATA_PATH in Cell 2 matches the exact path in Google Drive.
       - Google Drive paths are case-sensitive on Colab.

    3. 'No module named scripts' or 'No module named cyclelayer.data'
       - evaluate.py and run_louo.py add REPO_ROOT to sys.path automatically.
       - If still failing, add  sys.path.insert(0, str(REPO_ROOT))  at the top
         of the failing script and re-run.

    4. PH = None for all units
       - Expected at high stride (stride >= 50) or with few epochs.
       - With stride=5, >=50 epochs, GPU: model typically achieves PH for
         some units.  Use Val/pearson_r in TensorBoard to diagnose.
         If pearson_r > 0.9 but PH=None -> mid-trajectory spike problem.
         If pearson_r < 0.5               -> bias/collapse problem.

    5. Slow training
       - Confirm use_amp=True is active (shows in train log).
       - Reduce NUM_WORKERS if Colab shows DataLoader worker errors.
       - Use STRIDE_TRAIN = 10 or 50 for quick smoke tests.

    6. LOUO very slow
       - LOUO runs 6 full training loops (~6x single-run time).
       - Narrow with  --units 1 3  in run_louo.py to run only specific folds.

    7. Reproducing a run
       - The frozen config is stored in the run dir and zipped to Drive.
       - Load it: yaml.safe_load(open("config_frozen.yaml"))
       - Theta scaler: np.load("theta_scaler.npz")  -> mean, std arrays

    TENSORBOARD PANELS
    ==================

    Launch: Cell 12 (%tensorboard --logdir /content/runs)
    All panels are under the SCALARS tab.

    Loss/
      train_epoch, val_epoch  -- Total loss curve. Val > Train for many
                                 epochs = overfitting; val spiking = instability.
      train_rul_epoch         -- RUL component of training loss (MSE+asym).
      train_theta_epoch       -- Theta supervision component (Huber).
                                 If this dominates early on, increase warmup.
      val_rul_epoch           -- Val RUL loss. The most important loss for
                                 monitoring PH and RMSE progress.
      {k}_step                -- Per-step losses for dense monitoring.

    Val/                      [PLAUSIBILITY - check these first]
      mae                     -- Mean absolute error on val set. Should decrease
                                 smoothly. Sudden jump = instability.
      rmse                    -- Root MSE. Should track with mae.
      bias                    -- mean(pred - target). Positive = over-estimating
                                 RUL (model predicts longer life than actual).
                                 Watch: positive bias -> large S-score.
      pearson_r               -- Correlation between pred and target.
                                 ~ 0.95+ = model tracks RUL trend well.
                                 < 0.5   = model is near-constant.
                                 Sudden drop = collapse or spike.
      pred_mean, pred_std     -- Output distribution. pred_std -> 0 = collapse.
                                 pred_mean should track target_mean.
      pred_min, pred_max      -- Range check. Should span [0, max_rul].
      target_mean             -- Constant reference (sanity check).

    Train/
      mae, bias               -- Same stats for training set (per-batch approx).
                                 Large Train/mae vs Val/mae gap = overfitting.

    Theta/                    [PHYSICS / HEALTH PARAMS - cyclelayer_v1 only]
      mae_mean                -- Mean MAE across all 10 health parameters.
                                 Should decrease during the ramp phase.
      mae_fan_eff etc.        -- Per-channel MAE. Identifies which health
                                 parameters are hard to predict.
                                 Note: values are in normalised (z-score) space.
      pred_mean_{name}        -- Mean predicted value per channel (normalised).
                                 Should converge toward true_mean_{name}.
      true_mean_{name}        -- Mean true value per channel (constant ref).

    GradNorm/
      train_epoch             -- Mean pre-clip gradient L2 norm per epoch.
                                 > 10 for many epochs = LR too high or
                                 lambda_theta too large too early.
                                 Steadily near clip_norm = clipping is active;
                                 consider lowering LR.
      train_step              -- Per-step norm (noisy; use epoch for trend).

    Weights/
      param_norm              -- L2 norm of all model parameters.
                                 Should grow slowly and stabilise.
                                 Sudden jump = weight explosion (lower LR).

    Lambda/theta              -- Scheduled lambda_theta value per epoch.
                                 With 'delayed' schedule: 0 for warmup epochs,
                                 then ramp to lambda_theta_end.

    LR                        -- Learning rate (cosine decay).

    WHAT TO LOOK FOR
    ================

    Healthy training:
      Val/mae decreasing -> Val/pearson_r rising -> GradNorm stable -> Weights/param_norm stable

    Early instability (best checkpoint at epoch 3-5):
      GradNorm/train_epoch >> 1.0  in early epochs
      Fix: lower lr (3e-4), use 'delayed' schedule, increase warmup to 20 epochs

    Model collapse (PH=None, flat predictions):
      Val/pred_std -> 0, Val/pearson_r -> 0
      Fix: lower LR, check theta normalisation

    Over-estimation bias (large S-score):
      Val/bias > 0 persistently
      Fix: increase asymmetry weight in config, check asymmetry=0.5

    Theta supervision hurting RUL:
      Val/val_rul increasing while Val/theta_mae decreasing
      Fix: reduce lambda_theta_end or increase warmup_epochs
""").format(fallback=OOM_FALLBACK_BATCH)

print(NOTES)
